In [3]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path(r"D:\ai-fmcg-sales-insights\data\raw\m5")

print("is data file already exist：", DATA_PATH.exists())

for file in DATA_PATH.glob("*.csv"):
    print(f"{file.name}: {file.stat().st_size / 1024 / 1024:.2f} MB")

数据文件夹是否存在： True
calendar.csv: 0.10 MB
sales_train_evaluation.csv: 116.10 MB
sell_prices.csv: 193.97 MB


In [4]:
calendar = pd.read_csv(DATA_PATH / "calendar.csv")
sell_prices = pd.read_csv(DATA_PATH / "sell_prices.csv")
sales = pd.read_csv(DATA_PATH / "sales_train_evaluation.csv")

print("calendar:", calendar.shape)
print("sell_prices:", sell_prices.shape)
print("sales:", sales.shape)

calendar: (1969, 14)
sell_prices: (6841121, 4)
sales: (30490, 1947)


In [5]:
print("Calendar columns:")
print(calendar.columns.tolist())

print("\nSell prices columns:")
print(sell_prices.columns.tolist())

print("\nSales identifier columns:")
print(sales.columns[:6].tolist())

print("\nFirst and last sales day columns:")
print(sales.columns[6], "to", sales.columns[-1])

Calendar columns:
['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']

Sell prices columns:
['store_id', 'item_id', 'wm_yr_wk', 'sell_price']

Sales identifier columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

First and last sales day columns:
d_1 to d_1941


In [6]:
display(calendar.head())
display(sell_prices.head())
display(sales.iloc[:5, :12])

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,d_5,d_6
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0


In [7]:
print("Duplicate rows:")
print("calendar:", calendar.duplicated().sum())
print("sell_prices:", sell_prices.duplicated().sum())
print("sales:", sales.duplicated().sum())

print("\nMissing values:")
print("calendar:", calendar.isna().sum().sum())
print("sell_prices:", sell_prices.isna().sum().sum())
print("sales:", sales.isna().sum().sum())

print("\nBusiness dimensions:")
print("States:", sales["state_id"].unique())
print("Stores:", sales["store_id"].nunique())
print("Categories:", sales["cat_id"].unique())
print("Departments:", sales["dept_id"].unique())
print("Products:", sales["item_id"].nunique())


Duplicate rows:
calendar: 0
sell_prices: 0
sales: 0

Missing values:
calendar: 7542
sell_prices: 0
sales: 0

Business dimensions:
States: ['CA' 'TX' 'WI']
Stores: 10
Categories: ['HOBBIES' 'HOUSEHOLD' 'FOODS']
Departments: ['HOBBIES_1' 'HOBBIES_2' 'HOUSEHOLD_1' 'HOUSEHOLD_2' 'FOODS_1' 'FOODS_2'
 'FOODS_3']
Products: 3049


In [8]:
# Filter the sales data to the FOODS category
food_sales_wide = sales[sales["cat_id"] == "FOODS"].copy()

# Identify all daily sales columns
day_columns = [
    column for column in sales.columns
    if column.startswith("d_")
]

# Select the most recent 730 days
recent_day_columns = day_columns[-730:]

# Calculate the expected size after converting to long format
expected_rows = len(food_sales_wide) * len(recent_day_columns)

print("FOODS records:", len(food_sales_wide))
print(
    "Selected period:",
    recent_day_columns[0],
    "to",
    recent_day_columns[-1]
)
print("Number of selected days:", len(recent_day_columns))
print("Expected long-table rows:", expected_rows)

FOODS records: 14370
Selected period: d_1212 to d_1941
Number of selected days: 730
Expected long-table rows: 10490100


In [9]:
# Filter the calendar to the selected 730-day period
selected_calendar = calendar[
    calendar["d"].isin(recent_day_columns)
].copy()

# Convert the date column from text to datetime
selected_calendar["date"] = pd.to_datetime(
    selected_calendar["date"]
)

# Sort the calendar chronologically
selected_calendar = selected_calendar.sort_values("date")

print("Real date range:")
print(
    selected_calendar["date"].min(),
    "to",
    selected_calendar["date"].max()
)

print("Calendar records:", len(selected_calendar))

Real date range:
2014-05-24 00:00:00 to 2016-05-22 00:00:00
Calendar records: 730


In [10]:
# Review the selected FOODS records and calendar data
display(
    food_sales_wide[
        ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
    ].head()
)

display(
    selected_calendar[
        ["date", "wm_yr_wk", "weekday", "d", "event_name_1", "event_type_1"]
    ].head()
)

,id,item_id,dept_id,cat_id,store_id,state_id
1612,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA
1613,FOODS_1_002_CA_1_evaluation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA
1614,FOODS_1_003_CA_1_evaluation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA
1615,FOODS_1_004_CA_1_evaluation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA
1616,FOODS_1_005_CA_1_evaluation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA


,date,wm_yr_wk,weekday,d,event_name_1,event_type_1
1211,2014-05-24,11417,Saturday,d_1212,NaN,NaN
1212,2014-05-25,11417,Sunday,d_1213,NaN,NaN
1213,2014-05-26,11417,Monday,d_1214,MemorialDay,National
1214,2014-05-27,11417,Tuesday,d_1215,NaN,NaN
1215,2014-05-28,11417,Wednesday,d_1216,NaN,NaN


In [11]:
# Define the identifier columns
id_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

# Select one store for an initial transformation test
test_store_wide = food_sales_wide[
    food_sales_wide["store_id"] == "CA_1"
].copy()

print("Products in CA_1:", len(test_store_wide))

Products in CA_1: 1437


In [12]:
# Convert daily sales columns from wide format to long format
test_store_long = test_store_wide.melt(
    id_vars=id_columns,
    value_vars=recent_day_columns,
    var_name="d",
    value_name="units_sold"
)

print("Long-table shape:", test_store_long.shape)

display(test_store_long.head())

Long-table shape: (1049010, 8)


,id,item_id,dept_id,cat_id,store_id,state_id,d,units_sold
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1212,0
1,FOODS_1_002_CA_1_evaluation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1212,0
2,FOODS_1_003_CA_1_evaluation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,d_1212,0
3,FOODS_1_004_CA_1_evaluation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA,d_1212,14
4,FOODS_1_005_CA_1_evaluation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1212,0


In [13]:
# Select the calendar fields required for analysis
calendar_fields = [
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "d",
    "event_name_1",
    "event_type_1",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

# Add calendar and event information to each sales record
test_store_long = test_store_long.merge(
    selected_calendar[calendar_fields],
    on="d",
    how="left",
    validate="many_to_one"
)

print("Shape after calendar merge:", test_store_long.shape)
print(
    "Missing dates:",
    test_store_long["date"].isna().sum()
)

display(test_store_long.head())

Shape after calendar merge: (1049010, 19)
Missing dates: 0


,id,item_id,dept_id,cat_id,store_id,state_id,d,units_sold,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,snap_CA,snap_TX,snap_WI
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1212,0,2014-05-24,11417,Saturday,1,5,2014,NaN,NaN,0,0,0
1,FOODS_1_002_CA_1_evaluation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1212,0,2014-05-24,11417,Saturday,1,5,2014,NaN,NaN,0,0,0
2,FOODS_1_003_CA_1_evaluation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,d_1212,0,2014-05-24,11417,Saturday,1,5,2014,NaN,NaN,0,0,0
3,FOODS_1_004_CA_1_evaluation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA,d_1212,14,2014-05-24,11417,Saturday,1,5,2014,NaN,NaN,0,0,0
4,FOODS_1_005_CA_1_evaluation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1212,0,2014-05-24,11417,Saturday,1,5,2014,NaN,NaN,0,0,0


In [14]:
# Add the weekly selling price for each product and store
test_store_long = test_store_long.merge(
    sell_prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one"
)

# Calculate estimated sales revenue
test_store_long["sales_revenue"] = (
    test_store_long["units_sold"]
    * test_store_long["sell_price"]
)

print("Final test-table shape:", test_store_long.shape)
print(
    "Missing prices:",
    test_store_long["sell_price"].isna().sum()
)

display(
    test_store_long[
        [
            "date",
            "store_id",
            "item_id",
            "dept_id",
            "units_sold",
            "sell_price",
            "sales_revenue",
            "event_name_1"
        ]
    ].head(10)
)

Final test-table shape: (1049010, 21)
Missing prices: 17668


,date,store_id,item_id,dept_id,units_sold,sell_price,sales_revenue,event_name_1
0,2014-05-24,CA_1,FOODS_1_001,FOODS_1,0,2.24,0.00,NaN
1,2014-05-24,CA_1,FOODS_1_002,FOODS_1,0,8.88,0.00,NaN
2,2014-05-24,CA_1,FOODS_1_003,FOODS_1,0,2.88,0.00,NaN
3,2014-05-24,CA_1,FOODS_1_004,FOODS_1,14,1.78,24.92,NaN
4,2014-05-24,CA_1,FOODS_1_005,FOODS_1,0,3.28,0.00,NaN
5,2014-05-24,CA_1,FOODS_1_006,FOODS_1,0,2.24,0.00,NaN
6,2014-05-24,CA_1,FOODS_1_008,FOODS_1,0,3.28,0.00,NaN
7,2014-05-24,CA_1,FOODS_1_009,FOODS_1,0,2.48,0.00,NaN
8,2014-05-24,CA_1,FOODS_1_010,FOODS_1,0,4.74,0.00,NaN
9,2014-05-24,CA_1,FOODS_1_011,FOODS_1,0,2.48,0.00,NaN


In [15]:
# Check the size and completeness of the test dataset
print("Rows:", len(test_store_long))
print("Columns:", len(test_store_long.columns))

print(
    "Date range:",
    test_store_long["date"].min(),
    "to",
    test_store_long["date"].max()
)

print(
    "Unique products:",
    test_store_long["item_id"].nunique()
)

print(
    "Missing dates:",
    test_store_long["date"].isna().sum()
)

print(
    "Missing prices:",
    test_store_long["sell_price"].isna().sum()
)

print(
    "Rows with sales but no price:",
    (
        test_store_long["sell_price"].isna()
        & (test_store_long["units_sold"] > 0)
    ).sum()
)

Rows: 1049010
Columns: 21
Date range: 2014-05-24 00:00:00 to 2016-05-22 00:00:00
Unique products: 1437
Missing dates: 0
Missing prices: 17668
Rows with sales but no price: 0


In [16]:
# Flag whether the product had an available selling price
test_store_long["is_available"] = (
    test_store_long["sell_price"].notna()
)

# Create one state-specific SNAP indicator
test_store_long["snap_eligible"] = test_store_long["snap_CA"]

# Rename the source series identifier
test_store_long = test_store_long.rename(
    columns={"id": "series_id"}
)

# Select and order the final analytical fields
final_columns = [
    "date",
    "d",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "series_id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "units_sold",
    "sell_price",
    "sales_revenue",
    "is_available",
    "event_name_1",
    "event_type_1",
    "snap_eligible"
]

test_store_final = test_store_long[final_columns].copy()

display(test_store_final.head())
print("Final shape:", test_store_final.shape)

,date,d,wm_yr_wk,weekday,wday,month,year,series_id,item_id,dept_id,cat_id,store_id,state_id,units_sold,sell_price,sales_revenue,is_available,event_name_1,event_type_1,snap_eligible
0,2014-05-24,d_1212,11417,Saturday,1,5,2014,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,0,2.24,0.00,True,NaN,NaN,0
1,2014-05-24,d_1212,11417,Saturday,1,5,2014,FOODS_1_002_CA_1_evaluation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,0,8.88,0.00,True,NaN,NaN,0
2,2014-05-24,d_1212,11417,Saturday,1,5,2014,FOODS_1_003_CA_1_evaluation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,0,2.88,0.00,True,NaN,NaN,0
3,2014-05-24,d_1212,11417,Saturday,1,5,2014,FOODS_1_004_CA_1_evaluation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA,14,1.78,24.92,True,NaN,NaN,0
4,2014-05-24,d_1212,11417,Saturday,1,5,2014,FOODS_1_005_CA_1_evaluation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,0,3.28,0.00,True,NaN,NaN,0


Final shape: (1049010, 20)


In [17]:
# Create the processed-data output directory
PROJECT_ROOT = DATA_PATH.parents[2]
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed"

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

test_output_file = (
    OUTPUT_PATH
    / "fmcg_sales_CA_1.csv.gz"
)

# Export the transformed test dataset as a compressed CSV file
test_store_final.to_csv(
    test_output_file,
    index=False,
    compression="gzip",
    date_format="%Y-%m-%d"
)

print("Saved file:", test_output_file)
print(
    "File size:",
    round(
        test_output_file.stat().st_size
        / 1024
        / 1024,
        2
    ),
    "MB"
)

Saved file: D:\ai-fmcg-sales-insights\data\processed\fmcg_sales_CA_1.csv.gz
File size: 8.94 MB


In [18]:
import gc

# Define the identifier columns used during the wide-to-long transformation
id_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

# Define the calendar fields required for the analytical dataset
calendar_fields = [
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "d",
    "event_name_1",
    "event_type_1",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

# Define the final column order
final_columns = [
    "date",
    "d",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "series_id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "units_sold",
    "sell_price",
    "sales_revenue",
    "is_available",
    "event_name_1",
    "event_type_1",
    "snap_eligible"
]

# Get all store identifiers
store_ids = sorted(food_sales_wide["store_id"].unique())

processing_results = []

for store_id in store_ids:
    print(f"Processing {store_id}...")

    # Select the current store
    store_wide = food_sales_wide[
        food_sales_wide["store_id"] == store_id
    ].copy()

    # Convert daily sales from wide format to long format
    store_long = store_wide.melt(
        id_vars=id_columns,
        value_vars=recent_day_columns,
        var_name="d",
        value_name="units_sold"
    )

    # Add date, event and SNAP information
    store_long = store_long.merge(
        selected_calendar[calendar_fields],
        on="d",
        how="left",
        validate="many_to_one"
    )

    # Add weekly product prices
    store_long = store_long.merge(
        sell_prices,
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left",
        validate="many_to_one"
    )

    # Calculate estimated sales revenue
    store_long["sales_revenue"] = (
        store_long["units_sold"]
        * store_long["sell_price"]
    )

    # Flag whether the product was available for sale
    store_long["is_available"] = (
        store_long["sell_price"].notna()
    )

    # Select the correct state-specific SNAP indicator
    state_id = store_long["state_id"].iloc[0]
    snap_column = f"snap_{state_id}"

    store_long["snap_eligible"] = store_long[snap_column]

    # Rename the original series identifier
    store_long = store_long.rename(
        columns={"id": "series_id"}
    )

    # Select the final fields
    store_final = store_long[final_columns].copy()

    # Check for price anomalies
    sales_without_price = (
        store_final["sell_price"].isna()
        & (store_final["units_sold"] > 0)
    ).sum()

    # Define the output filename
    output_file = (
        OUTPUT_PATH
        / f"fmcg_sales_{store_id}.csv.gz"
    )

    # Export the store dataset as compressed CSV
    store_final.to_csv(
        output_file,
        index=False,
        compression="gzip",
        date_format="%Y-%m-%d"
    )

    file_size_mb = (
        output_file.stat().st_size
        / 1024
        / 1024
    )

    processing_results.append({
        "store_id": store_id,
        "rows": len(store_final),
        "missing_prices": int(
            store_final["sell_price"].isna().sum()
        ),
        "sales_without_price": int(sales_without_price),
        "file_size_mb": round(file_size_mb, 2)
    })

    print(
        f"Completed {store_id}: "
        f"{len(store_final):,} rows, "
        f"{file_size_mb:.2f} MB"
    )

    # Release memory before processing the next store
    del store_wide
    del store_long
    del store_final
    gc.collect()

print("All stores completed.")

Processing CA_1...
Completed CA_1: 1,049,010 rows, 8.94 MB
Processing CA_2...
Completed CA_2: 1,049,010 rows, 8.49 MB
Processing CA_3...
Completed CA_3: 1,049,010 rows, 9.20 MB
Processing CA_4...
Completed CA_4: 1,049,010 rows, 8.49 MB
Processing TX_1...
Completed TX_1: 1,049,010 rows, 8.64 MB
Processing TX_2...
Completed TX_2: 1,049,010 rows, 8.79 MB
Processing TX_3...
Completed TX_3: 1,049,010 rows, 8.76 MB
Processing WI_1...
Completed WI_1: 1,049,010 rows, 8.89 MB
Processing WI_2...
Completed WI_2: 1,049,010 rows, 8.99 MB
Processing WI_3...
Completed WI_3: 1,049,010 rows, 8.68 MB
All stores completed.


In [19]:
# Review the processing results for all stores
processing_summary = pd.DataFrame(processing_results)

display(processing_summary)

print(
    "Total rows:",
    f"{processing_summary['rows'].sum():,}"
)

print(
    "Total compressed size:",
    f"{processing_summary['file_size_mb'].sum():.2f} MB"
)

print(
    "Total rows with sales but no price:",
    processing_summary["sales_without_price"].sum()
)

,store_id,rows,missing_prices,sales_without_price,file_size_mb
0,CA_1,1049010,17668,0,8.94
1,CA_2,1049010,121842,0,8.49
2,CA_3,1049010,16541,0,9.20
3,CA_4,1049010,24381,0,8.49
4,TX_1,1049010,14616,0,8.64
5,TX_2,1049010,15484,0,8.79
6,TX_3,1049010,23338,0,8.76
7,WI_1,1049010,15302,0,8.89
8,WI_2,1049010,16863,0,8.99
9,WI_3,1049010,16457,0,8.68


Total rows: 10,490,100
Total compressed size: 87.87 MB
Total rows with sales but no price: 0


In [21]:
from pathlib import Path
import pandas as pd

# Use the absolute path to the processed files on drive D
DATA_PATH = Path(r"D:\ai-fmcg-sales-insights\data\processed")

stores = [
    "CA_1", "CA_2", "CA_3", "CA_4",
    "TX_1", "TX_2", "TX_3",
    "WI_1", "WI_2", "WI_3"
]

results = []

for store in stores:
    file_path = DATA_PATH / f"fmcg_sales_{store}.csv.gz"
    print(f"Reading: {file_path.name}")

    if not file_path.is_file():
        results.append({
            "file": file_path.name,
            "status": "MISSING",
            "rows": None,
            "columns": None,
            "size_mb": None
        })
        continue

    try:
        row_count = 0
        column_count = None

        # Read gzip-compressed CSV files in chunks
        for chunk in pd.read_csv(
            file_path,
            compression="gzip",
            chunksize=100_000
        ):
            row_count += len(chunk)
            column_count = len(chunk.columns)

        results.append({
            "file": file_path.name,
            "status": "OK",
            "rows": row_count,
            "columns": column_count,
            "size_mb": round(file_path.stat().st_size / (1024 ** 2), 2)
        })

    except Exception as error:
        results.append({
            "file": file_path.name,
            "status": f"ERROR: {error}",
            "rows": None,
            "columns": None,
            "size_mb": round(file_path.stat().st_size / (1024 ** 2), 2)
        })

# Summarize file integrity and export size
file_check = pd.DataFrame(results)
display(file_check)

print("Successfully read files:", (file_check["status"] == "OK").sum())
print("Total successfully read rows:", int(file_check["rows"].sum()))

Reading: fmcg_sales_CA_1.csv.gz


C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


Reading: fmcg_sales_CA_2.csv.gz


C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


Reading: fmcg_sales_CA_3.csv.gz


...

C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


Reading: fmcg_sales_WI_3.csv.gz


C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\17806\AppData\Local\Temp\ipykernel_26132\2232867621.py:34: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


,file,status,rows,columns,size_mb
0,fmcg_sales_CA_1.csv.gz,OK,1049010,20,8.94
1,fmcg_sales_CA_2.csv.gz,OK,1049010,20,8.49
2,fmcg_sales_CA_3.csv.gz,OK,1049010,20,9.20
3,fmcg_sales_CA_4.csv.gz,OK,1049010,20,8.49
4,fmcg_sales_TX_1.csv.gz,OK,1049010,20,8.64
5,fmcg_sales_TX_2.csv.gz,OK,1049010,20,8.79
6,fmcg_sales_TX_3.csv.gz,OK,1049010,20,8.76
7,fmcg_sales_WI_1.csv.gz,OK,1049010,20,8.89
8,fmcg_sales_WI_2.csv.gz,OK,1049010,20,8.99
9,fmcg_sales_WI_3.csv.gz,OK,1049010,20,8.68


Successfully read files: 10
Total successfully read rows: 10490100
